# 新闻标题能预测道琼斯指数涨跌吗？

**问题**：每天的世界新闻标题，真的能帮助预测道指（DJIA）第二天/当天的涨跌吗？如果有作用，它在统计上和经济上是真实的吗？

**用户**：一个刷到"AI 读新闻预测股市"广告的普通散户。

This project is a prototype and is not financial advice.

## 阶段 1：环境、数据清理、检查、切分、基准

### 1.0 导入与随机种子
为什么固定随机种子 42：之后的模型、bootstrap 都有随机性，固定种子才能让别人重跑得到一样的数字。

In [1]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 让 notebook 能 import 项目根目录下的 src/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import data_utils as du
from src.evaluation import evaluate, confusion_table

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA = ROOT / "data"
pd.set_option("display.max_colwidth", 120)

### 1.1 读取并清理新闻数据
- 原始标题很多长这样：`b'Georgia downs two Russian warplanes...'`，这是 Python 字节串被直接写进了 CSV，`b'` 和结尾的 `'` 不是标题内容，要去掉。
- 有少数格子是空的（NaN），换成空字符串，免得后面打情绪分时报错。
- `Date` 转成日期类型并排序，因为之后所有切分都按时间来。

In [2]:
news_raw = pd.read_csv(DATA / "Combined_News_DJIA.csv")
news = du.load_news(DATA / "Combined_News_DJIA.csv")

print("行数、列数:", news.shape)
print("日期范围:", news["Date"].min().date(), "到", news["Date"].max().date())
print("每列标题的空值数（清理前，只显示非零的）:")
na = news_raw[du.HEADLINE_COLS].isna().sum()
print(na[na > 0])

has_prefix = news_raw[du.HEADLINE_COLS].apply(lambda s: s.astype(str).str.match(r'^b[\'"]')).sum().sum()
left = news[du.HEADLINE_COLS].apply(lambda s: s.str.match(r'^b[\'"]')).sum().sum()
print(f"清理前带 b'' 前缀的标题: {has_prefix}，清理后仍带前缀: {left}")

行数、列数: (1989, 27)
日期范围: 2008-08-08 到 2016-07-01
每列标题的空值数（清理前，只显示非零的）:
Top23    1
Top24    3
Top25    3
dtype: int64
清理前带 b'' 前缀的标题: 11920，清理后仍带前缀: 0


In [3]:
# 清理前后对比，看看效果
print("清理前:", news_raw.loc[0, "Top1"])
print("清理后:", news.loc[0, "Top1"])

清理前: b"Georgia 'downs two Russian warplanes' as countries move to brink of war"
清理后: Georgia 'downs two Russian warplanes' as countries move to brink of war


### 1.2 读取道指价格并检查 Label
数据集作者说 Label = 1 表示道指"上涨或持平"。我们用价格自己算一遍，看看和 Label 是否一致。
同时检查两种可能的定义：**收盘价对前一天收盘价**、**收盘价对当天开盘价**。哪种一致率接近 100%，就说明 Label 实际是按哪种定义做的。

In [4]:
# Kaggle 下载下来的文件名是 upload_DJIA_table.csv，两种名字都接受
djia_path = next(p for p in [DATA / "DJIA_table.csv", DATA / "upload_DJIA_table.csv"] if p.exists())
print("价格文件:", djia_path.name)
djia = du.load_djia(djia_path)
print("道指表:", djia.shape, djia["Date"].min().date(), "到", djia["Date"].max().date())

# 两张表的日期是否一一对应
only_news = set(news["Date"]) - set(djia["Date"])
only_djia = set(djia["Date"]) - set(news["Date"])
print("只在新闻表里的日期数:", len(only_news), "| 只在道指表里的日期数:", len(only_djia))

merged, rates = du.check_labels(news, djia)
pd.DataFrame(rates).T

价格文件: upload_DJIA_table.csv
道指表: (1989, 7) 2008-08-08 到 2016-07-01
只在新闻表里的日期数: 0 | 只在道指表里的日期数: 0


,compared_days,agreement
up_adjclose,1988.0,0.998994
up_close,1988.0,0.998994
up_open_close,1989.0,0.983409


In [5]:
# 把不一致的日子列出来看看（用收盘对前一天收盘的定义）
mismatch = merged[merged["up_adjclose"].notna() & (merged["up_adjclose"] != merged["Label"])]
print("不一致的天数:", len(mismatch))
mismatch[["Date", "Label", "Open", "Close", "Adj Close", "up_adjclose", "up_open_close"]].head(15)

不一致的天数: 2


,Date,Label,Open,Close,Adj Close,up_adjclose,up_open_close
1436,2014-04-24,0,16503.390625,16501.650391,16501.650391,1.0,0.0
1839,2015-11-27,0,17806.039062,17813.390625,17813.390625,1.0,1.0


**看不一致那两天的前后几行**：如果收盘价和前一天一模一样，按"涨或持平 = 1"的规则应该是 1，但 Label 是 0。

In [6]:
for d in mismatch["Date"]:
    i = djia.index[djia["Date"] == d][0]
    print(djia.loc[i - 1:i, ["Date", "Open", "Close", "Volume"]].merge(news[["Date", "Label"]], on="Date"), "\n")

        Date          Open         Close    Volume  Label
0 2014-04-23  16513.730469  16501.650391  76830000      0
1 2014-04-24  16503.390625  16501.650391  80340000      0 

        Date          Open         Close    Volume  Label
0 2015-11-25  17820.810547  17813.390625  82540000      1
1 2015-11-27  17806.039062  17813.390625  82540000      0 



**结论**：Label 是按"今天收盘价 ≥ 前一天收盘价"定义的（一致率 99.9%）；"收盘 ≥ 开盘"只有 98.3%，不是它的定义。
（道指是指数，没有分红调整，所以 Close 和 Adj Close 完全一样，两行结果相同。）

仅有的 2 个不一致日（2014-04-24、2015-11-27），价格表里的收盘价和前一天**一模一样**，2015-11-27 连成交量都和前一天相同——很像价格表把前一天的数据复制了一行，是价格表的数据质量问题，而 Label 可能来自另一份价格。
处理方式：**预测目标仍然用原始 Label**（它是题目定义的目标，而且只差 2 天）。但要记住：2015-11-27 在测试集里，阶段 6 用价格算收益时，这一天的收益会被算成 0。

### 1.3 按日期切分
- 训练集：2008-08-08 到 2014-12-31
- 测试集：2015-01-02 到 2016-07-01

为什么按日期而不是随机打乱：现实中我们只能用过去预测未来。如果随机打乱，模型会"偷看"到未来的日子，结果会虚高。

In [7]:
train, test = du.split_by_date(news)
assert train["Date"].max() < test["Date"].min()  # 确认训练集完全在测试集之前
assert len(train) + len(test) == len(news)        # 确认没有日子被漏掉

split_summary = pd.DataFrame({
    "起始": [train["Date"].min().date(), test["Date"].min().date()],
    "结束": [train["Date"].max().date(), test["Date"].max().date()],
    "天数": [len(train), len(test)],
    "涨的比例": [train["Label"].mean(), test["Label"].mean()],
}, index=["训练集", "测试集"])
split_summary

,起始,结束,天数,涨的比例
训练集,2008-08-08,2014-12-31,1611,0.541899
测试集,2015-01-02,2016-07-01,378,0.507937


### 1.4 两个基准（只在测试集上算）
任何 AI 模型都必须先打败这两个"笨办法"，否则就没有价值：
1. **永远预测涨**：股市长期大多数日子在涨，所以这个准确率不会太低。
2. **跟前一天同方向**：检验"动量"——昨天涨今天也涨。测试集第一天用训练集最后一天的 Label。

指标：准确率为主；再看混淆矩阵和"跌"的召回率（真实下跌的日子里，猜中了几成）。永远预测涨的"跌"召回率必然是 0，这说明光看准确率会被骗。

In [8]:
y_test = test["Label"].to_numpy()
preds = {
    "永远预测涨": du.baseline_always_up(test),
    "跟前一天同方向": du.baseline_previous_day(train, test),
}

results = pd.DataFrame([evaluate(y_test, p, name) for name, p in preds.items()]).set_index("model")
results

,accuracy,down_recall
model,,
永远预测涨,0.507937,0.000000
跟前一天同方向,0.457672,0.451613


In [9]:
for name, p in preds.items():
    print(f"\n=== {name} ===")
    print(confusion_table(y_test, p))


=== 永远预测涨 ===
         预测 跌(0)  预测 涨(1)
真实 跌(0)        0      186
真实 涨(1)        0      192

=== 跟前一天同方向 ===
         预测 跌(0)  预测 涨(1)
真实 跌(0)       84      102
真实 涨(1)      103       89


### 阶段 1 小结
- 数据：1989 个交易日，每天 25 条标题；共 11920 条标题带 `b''` 前缀，已全部去掉；只有 7 个空标题（Top23–Top25），已换成空字符串。新闻表和价格表的日期完全对得上。
- 切分：训练集 1611 天（涨 54.2%），测试集 378 天（涨 50.8%）。注意测试期比训练期"更难"：涨的比例低了约 3.4 个百分点，所以训练时学到的"多数时候涨"在测试期没那么好用。
- 基准（测试集）：
  - 永远预测涨：准确率 **50.8%**，但"跌"召回率 **0%**——一次下跌都没抓到。
  - 跟前一天同方向：准确率 **45.8%**，比瞎猜还差；"跌"召回率 45.2%。说明 2015–2016 年道指日涨跌**没有动量**。低于 50% 看起来像"反转"，但 378 天里差 4 个百分点，完全可能是随机波动，不能当成规律。
- 这意味着：之后的新闻模型想说自己有用，至少要稳定超过 **50.8%**；而且因为测试集只有 378 天，几个百分点的差距可能只是运气——这正是阶段 5 要检验的。

## 阶段 2：VADER 情绪特征 + 逻辑回归

### 2.0 思路
"AI 读新闻"最简单的版本：先判断每条标题是正面还是负面，再看"今天新闻整体偏负面"能不能预测道指下跌。

**VADER** 是一个专为短文本（推文、标题）设计的情绪词典 + 规则模型：它给每个词一个情绪分，再按规则处理否定（"not good"）、程度词（"very"）、感叹号等，最后输出 compound 分，范围 -1（极负面）到 +1（极正面）。
优点：快、透明、不需要训练。缺点：它是为社交媒体通用情绪设计的，不懂金融语境（比如 "rates cut" 对股市是好事，VADER 可能看成负面）——阶段 3 的 FinBERT 就是为了解决这个问题。

注意：这一阶段和原始数据一样，用的是**当天**的标题预测**当天**的涨跌。这可能有"偷看"问题，阶段 4 专门检验。

### 2.1 给每条标题打分
打分是对每条标题独立进行的，不从数据里"学"任何东西，所以对全部日期一起打分不会造成训练/测试之间的泄露。

In [10]:
from src.sentiment import vader_scores, daily_features

vader = vader_scores(news)
print("打分表形状:", vader.shape, "| 空标题(NaN)数:", int(vader.isna().sum().sum()))

# 看几个例子：最负面、最正面、中性
flat = pd.DataFrame({"headline": news[du.HEADLINE_COLS].to_numpy().ravel(),
                     "compound": vader.to_numpy().ravel()}).dropna()
pd.concat([flat.nsmallest(3, "compound"), flat.nlargest(3, "compound"),
           flat[flat["compound"] == 0].head(3)])

打分表形状: (1989, 25) | 空标题(NaN)数: 7


,headline,compound
17551,"The war on drugs in Mexico, decreed by Caldern, part-funded by millions of dollars in U.S. govt aids, has not only f...",-0.9898
47737,"China strongly rejected U.S.-led criticism of its human rights record at the U.N. Human Rights Council, saying the U...",-0.9854
10136,"Everyone knows Japan fought against the ban that would have helped save the bluefin tuna from imminent collapse, but...",-0.9846
32026,"Snowden Nominated for Freedom of Thought Prize ""Members of the European Parliament are officially nominating fugitiv...",0.9623
14510,"""If the eurozone falls apart, it will be because Germany did not do enough to save it. If the eurozone is saved, it ...",0.9618
8270,It is a great honor for me to stand here tonight as the first Somali journalist to win this award. By recognizing me...,0.9608
1,BREAKING: Musharraf to be impeached.,0.0000
5,150 Russian tanks have entered South Ossetia whilst Georgia shoots down two Russian jets.,0.0000
13,China tells Bush to stay out of other countries' affairs,0.0000


In [11]:
# 所有标题的情绪分布：世界新闻整体偏负面
print(f"负面(<=-0.05): {(flat.compound <= -0.05).mean():.1%}  "
      f"中性: {flat.compound.between(-0.05, 0.05, inclusive='neither').mean():.1%}  "
      f"正面(>=0.05): {(flat.compound >= 0.05).mean():.1%}")

负面(<=-0.05): 54.5%  中性: 25.1%  正面(>=0.05): 20.4%


### 2.2 每天汇总成特征
一天 25 条标题 → 7 个数字：平均分、最低分、最高分、标准差、负面占比、正面占比、Top1 的分数。
下面**只在训练集里**看看：涨的日子和跌的日子，这些特征平均有没有区别。如果连训练集里都看不出差别，模型也很难学到东西。

In [12]:
vader_feat = daily_features(vader, "vader")
feat_cols = list(vader_feat.columns)
data2 = pd.concat([news[["Date", "Label"]], vader_feat], axis=1)
train2, test2 = du.split_by_date(data2)

X_train, y_train = train2[feat_cols].to_numpy(), train2["Label"].to_numpy()
X_test, y_test = test2[feat_cols].to_numpy(), test2["Label"].to_numpy()

train2.groupby("Label")[feat_cols].mean().T.rename(columns={0: "跌的日子", 1: "涨的日子"})

Label,跌的日子,涨的日子
vader_mean,-0.225751,-0.212552
vader_min,-0.886882,-0.887947
vader_max,0.646831,0.660955
vader_std,0.430366,0.434254
vader_neg_share,0.559783,0.543382
vader_pos_share,0.197236,0.205762
vader_top1,-0.197408,-0.175671


In [13]:
# 每个特征和 Label 的相关系数（仅训练集）。|r| 接近 0 = 几乎没有线性关系
train2[feat_cols].corrwith(train2["Label"]).round(3).to_frame("与 Label 的相关系数")

,与 Label 的相关系数
vader_mean,0.071
vader_min,-0.008
vader_max,0.042
vader_std,0.036
vader_neg_share,-0.077
vader_pos_share,0.052
vader_top1,0.024


### 2.3 在训练集内部调参（TimeSeriesSplit，5 折）
- 模型：标准化 + 逻辑回归。逻辑回归简单、可解释，每个特征一个系数，能看出"负面新闻多 → 更可能跌"是否成立。
- 调的参数：正则化强度 C，以及是否给"跌"类加权（class_weight）。
- 选参数只看训练集内部的交叉验证分数，**测试集完全不碰**。
- 同时算"永远预测涨"在同样 5 个验证折上的准确率，作为训练集内部的对照。

In [14]:
from src.modeling import make_logreg, tune, cv_table, always_up_cv_accuracy, LOGREG_GRID, CV

for i, (tr, va) in enumerate(CV.split(X_train)):
    print(f"折 {i+1}: 训练 {train2.Date[tr[0]].date()}~{train2.Date[tr[-1]].date()} ({len(tr)}天) | "
          f"验证 {train2.Date[va[0]].date()}~{train2.Date[va[-1]].date()} ({len(va)}天)")

search_vader = tune(make_logreg(), LOGREG_GRID, X_train, y_train)
cv_table(search_vader).head(6)

折 1: 训练 2008-08-08~2009-09-03 (271天) | 验证 2009-09-04~2010-09-28 (268天)
折 2: 训练 2008-08-08~2010-09-28 (539天) | 验证 2010-09-29~2011-10-19 (268天)
折 3: 训练 2008-08-08~2011-10-19 (807天) | 验证 2011-10-20~2012-11-13 (268天)
折 4: 训练 2008-08-08~2012-11-13 (1075天) | 验证 2012-11-14~2013-12-06 (268天)
折 5: 训练 2008-08-08~2013-12-06 (1343天) | 验证 2013-12-09~2014-12-31 (268天)


,param_logisticregression__C,param_logisticregression__class_weight,mean_test_score,std_test_score
0,0.001,NaN,0.553731,0.027277
1,0.01,NaN,0.548507,0.032271
2,0.1,NaN,0.531343,0.036908
3,1.0,NaN,0.528358,0.039531
4,0.01,balanced,0.526119,0.026279
5,0.001,balanced,0.524627,0.030824


In [15]:
best = search_vader.best_index_
fold_scores = pd.DataFrame({
    "VADER 逻辑回归(最佳参数)": [search_vader.cv_results_[f"split{i}_test_score"][best] for i in range(5)],
    "永远预测涨": always_up_cv_accuracy(y_train),
}, index=[f"折{i+1}" for i in range(5)])
fold_scores.loc["平均"] = fold_scores.mean()
print("最佳参数:", search_vader.best_params_)
fold_scores.round(3)

最佳参数: {'logisticregression__C': 0.001, 'logisticregression__class_weight': None}


,VADER 逻辑回归(最佳参数),永远预测涨
折1,0.582,0.575
折2,0.563,0.556
折3,0.504,0.500
折4,0.549,0.563
折5,0.571,0.560
平均,0.554,0.551


### 2.4 最终评估：测试集（参数已在上一步固定，此处只用一次）

In [16]:
vader_model = search_vader.best_estimator_
preds["VADER 逻辑回归"] = vader_model.predict(X_test)

results = pd.DataFrame([evaluate(y_test, p, name) for name, p in preds.items()]).set_index("model")
results["预测为涨的比例"] = [p.mean() for p in preds.values()]
results.round(3)

,accuracy,down_recall,预测为涨的比例
model,,,
永远预测涨,0.508,0.000,1.000
跟前一天同方向,0.458,0.452,0.505
VADER 逻辑回归,0.505,0.011,0.987


In [17]:
print(confusion_table(y_test, preds["VADER 逻辑回归"]))

         预测 跌(0)  预测 涨(1)
真实 跌(0)        2      184
真实 涨(1)        3      189


### 2.5 模型学到了什么？
系数是在标准化后的特征上算的，所以可以直接比大小：正系数 = 这个特征越大，模型越倾向预测"涨"。

In [18]:
coef = pd.Series(vader_model[-1].coef_[0], index=feat_cols).sort_values()
coef.round(4).to_frame("系数")

,系数
vader_neg_share,-0.0338
vader_min,-0.0087
vader_top1,0.0090
vader_std,0.0120
vader_max,0.0147
vader_pos_share,0.0151
vader_mean,0.0272


### 阶段 2 小结
- **情绪本身**：世界新闻整体偏负面（54.5% 标题为负面，只有 20.4% 正面）。在训练集里，涨的日子和跌的日子情绪特征几乎一样：每个特征和 Label 的相关系数绝对值都小于 0.08。方向倒是符合直觉（负面占比越高越容易跌，r = -0.077），但非常弱。
- **调参结果说明了问题**：交叉验证选中了正则化最强的 C = 0.001（网格里最小的值）。意思是"最好别太相信这些特征"，模型因此几乎总是预测涨。训练集内部 5 折平均 55.4%，"永远预测涨"是 55.1%，只高 0.3 个百分点，而且第 4 折还输了。
- **测试集**：VADER 逻辑回归准确率 **50.5%**，低于"永远预测涨"的 50.8%；98.7% 的日子都预测涨，"跌"的召回率只有 1.1%（186 个下跌日只抓到 2 个）。
- **系数**：负面占比的系数最负（-0.034），平均分的系数最正（+0.027），方向合理，但数值都很小，因为被强正则化压到了接近 0。
- **结论**：用 VADER 的通用情绪分，世界新闻标题**没有**表现出可用的预测能力，模型基本退化成了"永远预测涨"。可能的原因：(1) VADER 不懂金融语境；(2) r/worldnews 的热门标题大多是政治、社会新闻，和美国股市关系不大；(3) 每天的涨跌本来就接近随机。阶段 3 用金融专用的 FinBERT 检验原因 (1)。
- 说明：看完测试结果后，我只修了两个表格显示问题（参数 None 显示成 NaN、对照列没有对齐），**没有改任何模型或参数**。

## 阶段 3：FinBERT（金融专用情绪模型）

### 3.0 思路
阶段 2 的一个可能原因是 VADER 不懂金融语境。**FinBERT** 是在 BERT 语言模型基础上，用金融新闻句子（Financial PhraseBank）继续训练出来的情绪分类器，专门判断一句话对市场/公司是 positive、negative 还是 neutral。

- 每条标题得到三个概率（加起来 = 1），我们定义 **情绪分 = P(positive) − P(negative)**，范围 -1 到 +1，和 VADER 的 compound 可以直接对比。
- 这里只用现成模型做**推理**，不在我们的数据上训练，所以和阶段 2 一样不会造成训练/测试泄露。
- 约 5 万条标题在 CPU 上要算几十分钟，结果缓存到 `data/finbert_scores.csv`，之后直接读取。
- 每天的特征和阶段 2 完全一样（7 个），只是负面/正面占比改成按"概率最大的类别"来数。

In [19]:
from src.finbert import finbert_scores, to_wide

fb_long = finbert_scores(news, DATA / "finbert_scores.csv")
fb_long["score"] = fb_long["positive"] - fb_long["negative"]
fb_long["label"] = fb_long[["positive", "negative", "neutral"]].idxmax(axis=1)
print("打分的标题数:", len(fb_long))
fb_long["label"].value_counts(normalize=True).round(3).to_frame("FinBERT 标签占比")

从缓存读取: finbert_scores.csv
打分的标题数: 49718


,FinBERT 标签占比
label,
negative,0.506
neutral,0.436
positive,0.058


### 3.1 FinBERT 和 VADER 的看法一样吗？
如果两者几乎一样，换模型就没有意义；如果差别很大，就值得看看谁更"懂"金融。

In [20]:
# 把 FinBERT 结果和 VADER 结果按 (日期, 排名) 对齐
vader_long = vader.assign(Date=news["Date"]).melt(id_vars="Date", var_name="rank", value_name="vader")
vader_long["rank"] = vader_long["rank"].str.replace("Top", "").astype(int)
cmp = fb_long.merge(vader_long, on=["Date", "rank"])
cmp["vader_label"] = pd.cut(cmp["vader"], [-1.01, -0.05, 0.05 - 1e-9, 1.01],
                            labels=["negative", "neutral", "positive"])

print(f"单条标题情绪分的相关系数（FinBERT vs VADER）: {cmp['score'].corr(cmp['vader']):.3f}")
print(f"三分类标签一致的比例: {(cmp['label'] == cmp['vader_label'].astype(str)).mean():.1%}")
pd.crosstab(cmp["vader_label"], cmp["label"], normalize="index").round(3).rename_axis(index="VADER ↓ / FinBERT →")

单条标题情绪分的相关系数（FinBERT vs VADER）: 0.431
三分类标签一致的比例: 54.5%


label,negative,neutral,positive
VADER ↓ / FinBERT →,,,
negative,0.666,0.308,0.025
neutral,0.336,0.607,0.057
positive,0.289,0.566,0.146


In [21]:
# 分歧最大的例子：一个说很正面、另一个说很负面
flat_headlines = news.melt(id_vars="Date", value_vars=du.HEADLINE_COLS, var_name="rank", value_name="headline")
flat_headlines["rank"] = flat_headlines["rank"].str.replace("Top", "").astype(int)
cmp = cmp.merge(flat_headlines, on=["Date", "rank"])
cmp["gap"] = cmp["score"] - cmp["vader"]
show = ["headline", "vader", "score"]
pd.concat([cmp.nlargest(4, "gap")[show], cmp.nsmallest(4, "gap")[show]]).round(2).rename(columns={"score": "finbert"})

,headline,vader,finbert
19107,"Chile officially recognized 9,800 more Pinochet victims, increasing the total number of people killed, tortured or i...",-0.92,0.89
10228,"Ciudad Jurez is one of the fastest growing cities in the world, in spite of the fact that it is ""the most violent zo...",-0.90,0.90
46907,Violent crime rises by 27% after killings hit a five-year high in the UK,-0.92,0.84
11090,"Portugal unveils crisis tax to cut deficit - Jos Scrates, Portugals prime minister, on Thursday announced tough new ...",-0.95,0.78
7530,"For the past two years, Sudanese-born cellphone magnate Mo Ibrahim has given out $5 million prizes to outstanding Af...",0.94,-0.92
13674,Norway says China has canceled a ministerial meeting apparently in response to the award of the Nobel peace prize to...,0.88,-0.93
47927,Massive Bleaching Event Hits the Great Barrier Reef: Warm ocean temperatures have impacted large swaths of the Great...,0.84,-0.95
44074,Canadians pull the plug on renewable energy scheme - Canadian province of Nova Scotia cancels a successful project w...,0.85,-0.94


### 3.2 每天的 FinBERT 特征
和阶段 2 一样，先只在训练集里看涨的日子和跌的日子有没有区别。

In [22]:
fb_score_wide = to_wide(fb_long, "score")
fb_label_wide = to_wide(fb_long, "label")
assert len(fb_score_wide) == len(news)

fb_feat = daily_features(fb_score_wide, "finbert",
                         is_neg=fb_label_wide.eq("negative"), is_pos=fb_label_wide.eq("positive"))
fb_cols = list(fb_feat.columns)
data3 = pd.concat([news[["Date", "Label"]], vader_feat, fb_feat], axis=1)
train3, test3 = du.split_by_date(data3)

train3.groupby("Label")[fb_cols].mean().T.rename(columns={0: "跌的日子", 1: "涨的日子"}).round(4)

Label,跌的日子,涨的日子
finbert_mean,-0.3735,-0.3658
finbert_min,-0.9293,-0.9276
finbert_max,0.5880,0.5947
finbert_std,0.4453,0.4470
finbert_neg_share,0.5096,0.5012
finbert_pos_share,0.0498,0.0542
finbert_top1,-0.3368,-0.3421


In [23]:
train3[fb_cols].corrwith(train3["Label"]).round(3).to_frame("与 Label 的相关系数（训练集）")

,与 Label 的相关系数（训练集）
finbert_mean,0.039
finbert_min,0.032
finbert_max,0.012
finbert_std,0.014
finbert_neg_share,-0.039
finbert_pos_share,0.047
finbert_top1,-0.006


### 3.3 调参 + 测试
用和阶段 2 **完全相同**的流程（同样的网格、同样的 5 折时间序列交叉验证）训练两个模型：
1. 只用 FinBERT 的 7 个特征
2. VADER + FinBERT 一共 14 个特征（看两者合起来是否更好）

`run_experiment` 把"切分 → 训练集内调参 → 测试集评估一次"包成一个函数，保证做法一致。

In [24]:
from src.modeling import run_experiment

rows3 = {}
row, search_fb = run_experiment(data3, fb_cols, name="FinBERT 逻辑回归")
rows3["FinBERT"] = row
row, search_both = run_experiment(data3, feat_cols + fb_cols, name="VADER+FinBERT 逻辑回归")
rows3["VADER+FinBERT"] = row

best = search_fb.best_index_
fold_fb = pd.DataFrame({
    "FinBERT 逻辑回归(最佳参数)": [search_fb.cv_results_[f"split{i}_test_score"][best] for i in range(5)],
    "永远预测涨": always_up_cv_accuracy(train3["Label"].to_numpy()),
}, index=[f"折{i+1}" for i in range(5)])
fold_fb.loc["平均"] = fold_fb.mean()
print("FinBERT 最佳参数:", search_fb.best_params_)
print("VADER+FinBERT 最佳参数:", search_both.best_params_)
fold_fb.round(3)

FinBERT 最佳参数: {'logisticregression__C': 0.001, 'logisticregression__class_weight': None}
VADER+FinBERT 最佳参数: {'logisticregression__C': 0.001, 'logisticregression__class_weight': None}


,FinBERT 逻辑回归(最佳参数),永远预测涨
折1,0.549,0.575
折2,0.556,0.556
折3,0.500,0.500
折4,0.563,0.563
折5,0.560,0.560
平均,0.546,0.551


In [25]:
X_test3 = test3[fb_cols].to_numpy()
preds["FinBERT 逻辑回归"] = search_fb.best_estimator_.predict(X_test3)
preds["VADER+FinBERT 逻辑回归"] = search_both.best_estimator_.predict(test3[feat_cols + fb_cols].to_numpy())

results = pd.DataFrame([evaluate(y_test, p, name) for name, p in preds.items()]).set_index("model")
results["预测为涨的比例"] = [p.mean() for p in preds.values()]
results.round(3)

,accuracy,down_recall,预测为涨的比例
model,,,
永远预测涨,0.508,0.000,1.000
跟前一天同方向,0.458,0.452,0.505
VADER 逻辑回归,0.505,0.011,0.987
FinBERT 逻辑回归,0.508,0.000,1.000
VADER+FinBERT 逻辑回归,0.505,0.022,0.976


In [26]:
print(confusion_table(y_test, preds["FinBERT 逻辑回归"]))
pd.Series(search_fb.best_estimator_[-1].coef_[0], index=fb_cols).sort_values().round(4).to_frame("FinBERT 模型系数")

         预测 跌(0)  预测 涨(1)
真实 跌(0)        0      186
真实 涨(1)        0      192


,FinBERT 模型系数
finbert_neg_share,-0.0165
finbert_top1,-0.0061
finbert_max,-0.0005
finbert_std,0.0019
finbert_mean,0.0137
finbert_min,0.0155
finbert_pos_share,0.0240


### 阶段 3 小结
- **FinBERT 比 VADER 更"悲观"**：50.6% 的标题判为负面、43.6% 中性，只有 5.8% 正面（VADER 是 20.4% 正面）。
- **两个模型只是中度一致**：单条标题情绪分相关系数 0.43，三分类标签一致的只有 54.5%；VADER 判为正面的标题，FinBERT 只有 14.6% 同意。
- **分歧例子暴露了 FinBERT 的问题**：它是在财报、公司新闻上训练的，看到 rises / increasing / growing 这类"增长"词就倾向正面——"Violent crime rises by 27%" 被打成 +0.84，"Chile officially recognized 9,800 more Pinochet victims, increasing the total…" 被打成 +0.89。反过来，"canceled""pull the plug""impacted" 这种商业语境里的坏词会被打成很负面。它把"葡萄牙出台危机税以削减赤字"判成正面，倒是符合金融逻辑。**结论：FinBERT 懂金融语言，但不懂世界新闻**——换模型解决了一个问题，又带来了另一个。
- **特征依旧几乎没有区分度**：训练集里涨、跌日子的特征几乎一样，和 Label 的相关系数绝对值都 ≤ 0.047，比 VADER 还弱。
- **调参**：两个模型都又选中了最强正则化 C = 0.001。FinBERT 模型训练集 5 折平均 54.6%，**低于**"永远预测涨"的 55.1%。
- **测试集**：FinBERT 逻辑回归 100% 预测涨，结果和"永远预测涨"完全一样（50.8%）；VADER+FinBERT 合起来 50.5%，186 个下跌日只抓到 4 个。
- **结论**：阶段 2 提出的原因 (1)"VADER 不懂金融语境"**不是主要原因**——换成金融专用模型也没有信号。更可能的是原因 (2)（r/worldnews 的标题和美股关系不大）和原因 (3)（日涨跌本来就接近随机）。
- 说明：看完测试结果后没有修改任何模型、参数或特征。

## 阶段 4：是不是在"偷看"？（同一天 vs 前一天的新闻）

### 4.0 问题在哪
原始数据用**当天**的标题预测**当天**的涨跌（收盘对前一天收盘）。可是：
- 这些标题是 r/worldnews 一整天的热门帖子，美股下午 4 点（美东）就收盘了，之后发的帖子**已经知道当天涨跌**。
- 有的标题本身就在报道市场（"Stocks plunge as …"），这等于把答案写进了题目。

真实的散户在开盘前只能看到**昨天**的新闻。所以要比较三种设定：

| 设定 | 特征 | 目标 | 能不能真的用来交易？ |
|---|---|---|---|
| A 同一天 | 第 t 天标题 | 第 t 天 收盘 ≥ 第 t−1 天收盘（原始 Label） | 不能，有"偷看"风险 |
| B 前一天 | 第 t−1 天标题 | 第 t 天 收盘 ≥ 第 t−1 天收盘 | 大部分可以，但第 t−1 天收盘后的新闻也算进了特征，而它影响的隔夜跳空也在目标里 |
| C 前一天 → 今天盘中 | 第 t−1 天标题 | 第 t 天 收盘 ≥ 第 t 天开盘 | **可以**：开盘时昨天的新闻全都已知，开盘买入、收盘卖出 |

如果 A 明显好于 B、C，说明 A 的"效果"来自偷看，不是预测。

### 4.1 直接找证据：标题里有没有在报道股市？

In [27]:
import re

MARKET_WORDS = re.compile(r"\b(?:stocks?|stock markets?|dow|wall street|s&p|nasdaq|shares)\b", re.IGNORECASE)
DOWN_WORDS = re.compile(r"\b(?:plunge[sd]?|crash(?:es|ed)?|tumble[sd]?|fall(?:s|en)?|fell|slump(?:s|ed)?|sink[s]?|sank|drop(?:s|ped)?|lose[s]?|lost|sell-?off)\b", re.IGNORECASE)
UP_WORDS = re.compile(r"\b(?:rall(?:y|ies|ied)|surge[sd]?|soar(?:s|ed)?|jump(?:s|ed)?|rise[sn]?|rose|gain(?:s|ed)?|climb(?:s|ed)?|rebound(?:s|ed)?)\b", re.IGNORECASE)

hl = flat_headlines[flat_headlines["headline"] != ""].copy()
hl["market"] = hl["headline"].str.contains(MARKET_WORDS)
mk = hl[hl["market"]].copy()
mk["说跌"] = mk["headline"].str.contains(DOWN_WORDS)
mk["说涨"] = mk["headline"].str.contains(UP_WORDS)
mk = mk.merge(news[["Date", "Label"]], on="Date")
print(f"提到股市的标题: {len(mk)} 条（占全部 {len(mk) / len(hl):.1%}），分布在 {mk['Date'].nunique()} 天")

only_down = mk[mk["说跌"] & ~mk["说涨"]]
only_up = mk[mk["说涨"] & ~mk["说跌"]]
pd.DataFrame({
    "标题数": [len(only_down), len(only_up), len(mk), len(hl)],
    "当天真的涨的比例": [only_down["Label"].mean(), only_up["Label"].mean(), mk["Label"].mean(),
                    hl.merge(news[["Date", "Label"]], on="Date")["Label"].mean()],
}, index=["提到股市 + 只说跌", "提到股市 + 只说涨", "所有提到股市的", "全部标题（对照）"]).round(3)

提到股市的标题: 185 条（占全部 0.4%），分布在 162 天


,标题数,当天真的涨的比例
提到股市 + 只说跌,38,0.263
提到股市 + 只说涨,2,1.000
所有提到股市的,185,0.400
全部标题（对照）,49718,0.535


In [28]:
# 举几个例子：标题说股市跌，当天 Label 是多少？
only_down.sort_values("Date")[["Date", "headline", "Label"]].head(8)

,Date,headline,Label
15,2008-09-15,EU Stocks plunge!,0
123,2008-09-17,Russian stock market falls 17%,0
24,2008-09-17,"Russian stock exchanges halt trades, drop 60% since May",0
61,2008-10-06,Biggest ever fall in UK stocks,0
162,2008-10-13,Bank shares fall despite bail-out,1
168,2010-04-28,"Greece bans short-selling as panic spreads, Shares tumble across Britain, Europe and Asia after the cost of insuring...",1
99,2011-06-06,The Revolution Blows Up --- \nEgypts stock market is tanking and its rich are taking their money to Zurich. Will an ...,0
17,2011-07-13,Rupert Murdoch is bleeding cash trying to save plummeting News Corp: The growing phone hacking scandal has sent News...,1


### 4.2 三种设定 × 两种情绪特征
- B、C 的做法：把每天的特征**整体往后挪一天**（第 t 行放第 t−1 天的特征），第一天没有"前一天"，删掉。测试集第一天用的是训练集最后一天的新闻，那是过去的信息，没问题。
- C 的目标用道指价格算：当天收盘 ≥ 当天开盘 = 1。
- 每个组合都跑一遍 `run_experiment`（同样的调参、同样的切分），并且列出"永远预测涨"在同一目标上的准确率作对照——三种目标涨的比例不一样，**只能和同一目标的基准比**。

In [29]:
feature_sets = {"VADER": feat_cols, "FinBERT": fb_cols}

base = data3.merge(djia[["Date", "Open", "Close"]], on="Date")
base["up_intraday"] = (base["Close"] >= base["Open"]).astype(int)

prev = base.copy()
prev[feat_cols + fb_cols] = prev[feat_cols + fb_cols].shift(1)   # 第 t 行 ← 第 t-1 天的新闻特征
prev = prev.dropna(subset=feat_cols + fb_cols).reset_index(drop=True)

settings = {
    "A 同一天新闻 → 当天涨跌": (base, "Label"),
    "B 前一天新闻 → 当天涨跌": (prev, "Label"),
    "C 前一天新闻 → 当天盘中涨跌": (prev, "up_intraday"),
}

rows4 = []
for s_name, (df, target) in settings.items():
    for f_name, cols in feature_sets.items():
        row, _ = run_experiment(df, cols, label_col=target, name=f_name)
        row["设定"] = s_name
        rows4.append(row)

table4 = pd.DataFrame(rows4).set_index(["设定", "model"])
table4["测试集 比基准"] = table4["accuracy"] - table4["test_always_up"]
table4[["cv_accuracy", "cv_always_up", "accuracy", "test_always_up", "测试集 比基准",
        "down_recall", "pred_up_share", "best_C", "class_weight"]].round(3)

cv_accuracy  cv_always_up  accuracy  test_always_up  \
设定               model                                                          
A 同一天新闻 → 当天涨跌   VADER          0.554         0.551     0.505           0.508   
                 FinBERT        0.546         0.551     0.508           0.508   
B 前一天新闻 → 当天涨跌   VADER          0.544         0.551     0.508           0.508   
                 FinBERT        0.534         0.551     0.508           0.508   
C 前一天新闻 → 当天盘中涨跌 VADER          0.554         0.557     0.516           0.516   
                 FinBERT        0.547         0.557     0.516           0.516   

                          测试集 比基准  down_recall  pred_up_share  best_C  \
设定               model                                                  
A 同一天新闻 → 当天涨跌   VADER     -0.003        0.011          0.987   0.001   
                 FinBERT    0.000        0.000          1.000   0.001   
B 前一天新闻 → 当天涨跌   VADER      0.000        0.000          1.000   0.001   
                 FinBERT    0.000        0.000          1.000   0.001   
C 前一天新闻 → 当天盘中涨跌 VADER      0.000        0.000          1.000   0.001   
                 FinBERT    0.000        0.000          1.000   0.001   

                         class_weight  
设定               model                 
A 同一天新闻 → 当天涨跌   VADER           None  
                 FinBERT         None  
B 前一天新闻 → 当天涨跌   VADER           None  
                 FinBERT         None  
C 前一天新闻 → 当天盘中涨跌 VADER           None  
                 FinBERT         None

In [30]:
# 训练集里：每个特征和目标的相关系数，三种设定放在一起比
corr4 = pd.DataFrame({
    s_name: du.split_by_date(df)[0][feat_cols + fb_cols].corrwith(du.split_by_date(df)[0][target])
    for s_name, (df, target) in settings.items()
})
corr4.round(3)

,A 同一天新闻 → 当天涨跌,B 前一天新闻 → 当天涨跌,C 前一天新闻 → 当天盘中涨跌
vader_mean,0.071,0.005,0.010
vader_min,-0.008,0.028,0.024
vader_max,0.042,0.031,0.032
vader_std,0.036,0.029,0.036
vader_neg_share,-0.077,-0.010,-0.016
vader_pos_share,0.052,0.005,0.012
vader_top1,0.024,-0.028,-0.019
finbert_mean,0.039,0.029,0.031
finbert_min,0.032,0.041,0.045
finbert_max,0.012,-0.005,-0.002


### 阶段 4 小结
- **"偷看"的直接证据确实存在**：有 185 条标题在报道股市，其中 38 条只说"跌"（如 2008-09-15 雷曼破产当天的 "EU Stocks plunge!"、2008-09-17 "Russian stock market falls 17%"），这些日子道指当天真的涨的比例只有 **26.3%**，远低于全部标题对应的约 53.5%。也就是说，同一天的新闻里有一部分**已经知道了答案**——它们报道的是当天的全球行情，而全球股市同一天大多同涨同跌。
- **但规模很小**：提到股市的标题只占 0.4%，分布在 162 天里，所以不会让模型"看起来很准"。
- **换成前一天的新闻，那点微弱的相关性也消失了**：同一天设定（A）里最强的两个特征是 VADER 平均分（r = 0.071）和负面占比（r = −0.077）；换成前一天的新闻（B）后变成 0.005 和 −0.010。训练集有 1611 天，相关系数纯粹因为随机就会在大约 ±0.05（两倍标准误）之内波动，所以 A 里的 0.077 只是勉强超出噪声，B、C 里的所有特征都完全在噪声范围内。阶段 2 看到的"方向正确但很弱"的信号，**很可能来自同一天对行情的报道，而不是预测能力**（阶段 5 会正式检验）。
- **模型层面**：6 个组合（3 种设定 × 2 种情绪）全部选中 C = 0.001，在测试集上几乎都退化成"永远预测涨"，比同一目标的基准低 0.3 个百分点到持平；训练集内部也只有 A-VADER 比基准高 0.3 个百分点，其余都持平或更低。
- **结论**：去掉"偷看"后，**世界新闻标题的情绪对第二天道指涨跌没有可检测的预测能力**。之后的阶段 5、6 以设定 C（前一天新闻 → 当天盘中，开盘就能操作）作为"真实可用"的版本，设定 A 只作为对照。